### 总结中间件 SummarizationMiddleware

当对话历史变长、接近模型上下文窗口时，`SummarizationMiddleware` 会把较早的消息自动总结成一条摘要消息，从而压缩上下文，同时保留最近的一些消息。

- `trigger`：触发压缩的条件，支持多种写法，满足任意一条即触发（列表是 **OR**；同一个 `dict` 里的多个条件是 **AND**）
  - `("tokens", 100)`：估算 token 数 ≥ 100
  - `("messages", 6)`：消息条数 ≥ 6
  - `("fraction", 0.001)`：达到模型最大输入窗口的 0.1%
- `keep`：压缩后保留多少最近的消息（不支持多个值）

> 注意：`fraction` 需要模型提供上下文窗口大小（`profile["max_input_tokens"]`），DeepSeek 默认没有该信息，所以要显式传入 `profile`。

#### `profile` 里的 `max_input_tokens` 是谁的？

`profile` 是挂在**模型对象**上的元数据，用来描述该模型的真实上下文窗口。

- 本示例中同一个 `model` 既负责回答、又被传给 `SummarizationMiddleware` 做总结，所以「回答模型」和「总结模型」是同一个，窗口也是同一个值。
- `fraction` 的阈值是按**传给中间件的那个模型**的 `profile` 计算的；如果用另一个模型专门做总结，要注意它的窗口可能不同。
- 这个值**不会改变模型真实窗口**，只是让 LangChain 组件知道边界，填错只会误导组件判断。
- **只有用 `fraction` 才需要 `profile`**，`("tokens", …)` / `("messages", …)` 不需要。

如何确定窗口大小：优先查服务商文档；LangChain 1.1+ 若能自动读到 `model.profile` 就无需手填，读不到（如本示例的 DeepSeek）时才需要手动传入。

In [5]:
## trigger， keep 参数
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage

load_dotenv(override=True)

# 关闭思考模式；并显式给出模型上下文窗口，fraction 触发条件才可用
model = init_chat_model(
    api_base=os.getenv("DEEPSEEK_API_BASE"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    model="deepseek-flash",
    model_provider="deepseek",
    # model_kwargs={"reasoning_effort": "none"},
    profile={"max_input_tokens": 65536},
)

# trigger 列表里任意一条满足即压缩：tokens>=100 或 messages>=6 或 达到窗口的 0.1%
# keep=("messages", 2) 表示压缩后只保留最近 2 条消息
agent = create_agent(
    model="deepseek:deepseek-flash",
    middleware=[
        SummarizationMiddleware(
            model,
            trigger=[
                ("tokens", 100),
                ("messages", 6),
                ("fraction", 0.001),
            ],
            keep=("messages", 2),
        )
    ],
)


In [ ]:
# 查看模型 profile：只有能提供 max_input_tokens，fraction 触发才可用
print("带 profile 的模型：", model.profile)

# 不传 profile 时，构造带 fraction 的中间件会直接报错
model_no_profile = init_chat_model(
    api_base=os.getenv("DEEPSEEK_API_BASE"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    model="deepseek-flash",
    model_provider="deepseek",
)
try:
    SummarizationMiddleware(model_no_profile, trigger=[("fraction", 0.001)])
except ValueError as e:
    print("缺少 profile 报错：", str(e).splitlines()[0])


#### 构造较长的对话历史并运行

In [6]:
# 构造一段较长的对话历史（7 轮问答，共 13 条消息），足以触发压缩
messages = [
    HumanMessage(content="你好，我想学习 LangChain。"),
    AIMessage(content="好的，LangChain 是一个用于构建大模型应用的框架，支持模型、提示词、工具、链等。"),
    HumanMessage(content="那第一个要学什么？"),
    AIMessage(content="建议先学模型初始化与调用，也就是 ChatModel 的基本用法。"),
    HumanMessage(content="模型调用有哪些方式？"),
    AIMessage(content="常见有 invoke、batch、stream 三种，分别对应同步、批量和流式。"),
    HumanMessage(content="流式有什么用？"),
    AIMessage(content="流式可以边生成边显示，提升交互体验，适合聊天场景。"),
    HumanMessage(content="那工具调用呢？"),
    AIMessage(content="工具调用可以让模型决定调用外部函数，比如查询天气、搜索网页等。"),
    HumanMessage(content="那中间件又是做什么的？"),
    AIMessage(content="中间件可以在模型调用前后插入逻辑，比如压缩上下文、注入内容、拦截工具等。"),
    HumanMessage(content="最后帮我总结一下学习路线。"),
]

# 运行 Agent：中间件会在调用模型前检查历史，满足 trigger 就先压缩
result = agent.invoke({"messages": messages})

# 压缩后会插入一条带 lc_source=summarization 标记的摘要消息，用它来判断是否发生了压缩
compressed = any(
    getattr(m, "additional_kwargs", {}).get("lc_source") == "summarization"
    for m in result["messages"]
)
if compressed:
    print("上下文已压缩")

print("压缩后消息数：", len(result["messages"]))
print("压缩后消息类型：", [type(m).__name__ for m in result["messages"]])
print(result["messages"])


上下文已压缩
压缩后消息数： 4
压缩后消息类型： ['HumanMessage', 'AIMessage', 'HumanMessage', 'AIMessage']
[HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\n用户正在学习 LangChain，希望获得循序渐进的学习指导。当前学习进度：已了解 LangChain 基本定位、模型初始化/调用、invoke/batch/stream、流式的用途、工具调用；最新问题是“中间件又是做什么的？”需要接下来解释 LangChain 中间件的作用。\n\n## SUMMARY\n\n- 用户目标：学习 LangChain，从基础开始逐步理解核心概念。\n- 已确定的学习顺序：\n  1. 先学模型初始化与调用，即 ChatModel 基本用法，因为这是基础。\n  2. 再学模型调用方式。\n- LangChain 被介绍为用于构建大模型应用的框架，支持模型、提示词、工具、链等。\n- 模型调用常见方式：\n  - `invoke`：同步调用\n  - `batch`：批量调用\n  - `stream`：流式调用\n- 流式调用的作用：边生成边显示，提升交互体验，适合聊天场景。\n- 工具调用的作用：让模型决定调用外部函数，例如查询天气、搜索网页等。\n- 用户最后提出：中间件是做什么的？该问题尚未回答。\n- 未出现明确被拒绝的方案或分歧。\n- 关键策略：按概念顺序教学，先模型调用，再工具调用，再进入中间件等更进阶主题。\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\n回答用户当前问题：解释 LangChain 中间件（middleware）是做什么的。应结合已学内容说明中间件通常用于拦截、包装或增强模型/代理/链的执行流程，例如日志记录、重试、限流、权限校验、动态提示词/工具、人在回路、输入输出处理等，并说明它与模型调用、工具调用、Agent/链之间的关系。回答后可根据用户反馈继续下一个 LangChain 学习主题。', additional_kwargs={'lc_source': 'summarization'}, response

#### 查看压缩后的历史

In [7]:
# 压缩后的第一条就是生成的摘要，后面是保留的最近消息
print("========== 摘要（压缩后的上下文）==========")
print(result["messages"][0].content)

print("\n========== 保留的最近消息 ==========")
for m in result["messages"][1:]:
    print(f"[{type(m).__name__}]", str(m.content))


========== 摘要（压缩后的上下文）==========
Here is a summary of the conversation to date:

## SESSION INTENT

用户正在学习 LangChain，希望获得循序渐进的学习指导。当前学习进度：已了解 LangChain 基本定位、模型初始化/调用、invoke/batch/stream、流式的用途、工具调用；最新问题是“中间件又是做什么的？”需要接下来解释 LangChain 中间件的作用。

## SUMMARY

- 用户目标：学习 LangChain，从基础开始逐步理解核心概念。
- 已确定的学习顺序：
  1. 先学模型初始化与调用，即 ChatModel 基本用法，因为这是基础。
  2. 再学模型调用方式。
- LangChain 被介绍为用于构建大模型应用的框架，支持模型、提示词、工具、链等。
- 模型调用常见方式：
  - `invoke`：同步调用
  - `batch`：批量调用
  - `stream`：流式调用
- 流式调用的作用：边生成边显示，提升交互体验，适合聊天场景。
- 工具调用的作用：让模型决定调用外部函数，例如查询天气、搜索网页等。
- 用户最后提出：中间件是做什么的？该问题尚未回答。
- 未出现明确被拒绝的方案或分歧。
- 关键策略：按概念顺序教学，先模型调用，再工具调用，再进入中间件等更进阶主题。

## ARTIFACTS

None

## NEXT STEPS

回答用户当前问题：解释 LangChain 中间件（middleware）是做什么的。应结合已学内容说明中间件通常用于拦截、包装或增强模型/代理/链的执行流程，例如日志记录、重试、限流、权限校验、动态提示词/工具、人在回路、输入输出处理等，并说明它与模型调用、工具调用、Agent/链之间的关系。回答后可根据用户反馈继续下一个 LangChain 学习主题。

========== 保留的最近消息 ==========
[AIMessage] 中间件可以在模型调用前后插入逻辑，比如压缩上下文、注入内容、拦截工具等。
[HumanMessage] 最后帮我总结一下学习路线。
[AIMessage] ## LangChain 学习路线总结

### 1. 基础认知（已完成）
- Lan

### 中间件内部是怎么工作的

每次调用模型前，`SummarizationMiddleware` 的 `before_model` 钩子会执行：

1. 给缺少 id 的消息补上唯一 id。
2. 用 `token_counter` 估算当前历史的 token 数，再由 `_should_summarize` 判断是否触发：
   - `trigger` 列表之间是 **OR**，同一个 `dict` 内多个条件是 **AND**；
   - `messages` 比对消息条数；`tokens`/`fraction` 既看估算值，也看上一条 AI 消息的**真实用量** `usage_metadata.total_tokens`；
   - `fraction` 的阈值 = `max_input_tokens × fraction`（本例 65536 × 0.001 ≈ 65）。
3. 触发后由 `_determine_cutoff_index` 依据 `keep` 计算切点，并保证不拆散 `AI(tool_calls)` 与 `ToolMessage` 的配对。
4. `_create_summary` 调用 `self._summary_model.invoke(...)` 生成摘要。
5. 返回新状态：`[RemoveMessage(REMOVE_ALL_MESSAGES), 摘要消息, *保留的消息]`。摘要是一条 `HumanMessage`，其 `additional_kwargs` 带 `lc_source="summarization"`——这就是我们用来判断「上下文已压缩」的标记。

一句话总结：**窗口大小由你通过 `profile` 告诉 LangChain；触发判断由「估算 token + 真实 usage」共同决定；`fraction` 只是把你的窗口按比例换算成阈值。**